In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# ====================================================
# 1. SETUP & CHARGEMENT DU "CERVEAU" (Modèle Sauvegardé)
# ====================================================

%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install sentence-transformers faiss-cpu pandas bert-score rouge_score evaluate

from unsloth import FastLanguageModel
from google.colab import drive
import torch

# A. Connexion au Drive
drive.mount('/content/drive')

# 👇 CHEMIN EXACT DE TON MODÈLE SAUVEGARDÉ (Vérifie ce chemin !)
MODEL_PATH = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Hosni Youssef/Modele_CNRS_FineTuned_QWEN"

print(f"⏳ Chargement du modèle depuis : {MODEL_PATH}...")

# B. Chargement (Mode Inférence = Rapide)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

print("✅ Modèle chargé ! Prêt à être testé.")

KeyError: 'grpo_selective_log_softmax'

In [ ]:
# ==================================================================
# 🚀 SYSTÈME RAG AVANCÉ (Reranking + Orientation + Fine-Tuned Model)
# ==================================================================
import json
import torch
import faiss
import numpy as np
import os
from sentence_transformers import SentenceTransformer, CrossEncoder

# --- 1. CONFIGURATION ---
JSON_PATH = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/concours_data.json"
TOP_K = 3                 # Nombre de documents finaux
SIMILARITY_THRESHOLD = 0.3 # Seuil de pertinence
USE_RERANKER = True       # Activer le tri précis

# --- 2. CHARGEMENT DES OUTILS (Embeddings & Reranker) ---
print("⚙️ Chargement du modèle d'Embeddings et du Reranker...")

# Modèle de recherche rapide (Vectoriel)
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

# Modèle de tri précis (Reranker) - Issu de ton notebook
reranker = None
if USE_RERANKER:
    try:
        reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)
        print("✅ Reranker activé (Cross-Encoder).")
    except Exception as e:
        print(f"⚠️ Reranker non chargé (Erreur: {e}). On utilisera FAISS seul.")

# --- 3. CHARGEMENT DES DONNÉES & INDEX FAISS ---
print(f"📚 Chargement des documents depuis : {JSON_PATH}...")
with open(JSON_PATH, 'r') as f:
    documents = json.load(f)

print(f"🔹 {len(documents)} chunks chargés. Création de l'index...")
texts = [d["text"] for d in documents]
embeddings = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("✅ Index de recherche prêt !")

# --- 4. FONCTIONS INTELLIGENTES (Issues de ton Notebook) ---

def handle_small_talk(question: str):
    """Gère la politesse sans appeler le modèle"""
    q = question.lower().strip()
    if q in ["bonjour", "bonsoir", "salut", "hello"]:
        return "Bonjour 👋 Je suis l’assistant concours CNRS. Comment puis-je vous aider ?"
    if q in ["merci", "top", "thanks"]:
        return "Avec plaisir ! N'hésitez pas si vous avez d'autres questions."
    return None

def is_orientation_question(question: str) -> bool:
    """Détecte si c'est une demande d'orientation/profil"""
    keywords = ["profil", "je suis", "quel concours", "orienter", "adapté", "accessible avec", "mon cv"]
    return any(k in question.lower() for k in keywords)

def build_search_query(question: str) -> str:
    """Optimise la recherche pour les mots-clés techniques"""
    q = question.lower()
    keywords = []
    if "statistique" in q or "data" in q: keywords.append("analyse de données big data")
    if "bio" in q or "vivant" in q: keywords.append("sciences biologiques")
    if "info" in q or "dev" in q: keywords.append("informatique logiciel")

    return " ".join(keywords) if keywords else question

def retrieve(question: str):
    """Recherche Hybride : FAISS (Large) + CrossEncoder (Précis)"""
    # A. Recherche large (x3 candidats)
    query_vec = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_vec, k=TOP_K * 3)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(documents): continue
        if score < SIMILARITY_THRESHOLD: continue # Filtre de bruit
        doc = documents[idx].copy()
        doc["faiss_score"] = float(score)
        candidates.append(doc)

    if not candidates: return []

    # B. Reranking (Tri intelligent)
    if reranker:
        pairs = [(question, c["text"]) for c in candidates]
        rerank_scores = reranker.predict(pairs)
        for c, r_score in zip(candidates, rerank_scores):
            c["rerank_score"] = float(r_score)
        # On trie par score CrossEncoder (plus fiable)
        candidates = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)

    return candidates[:TOP_K]

# --- 5. FONCTION PRINCIPALE : ANSWER (Connectée au Fine-Tuning) ---

def answer(question: str):
    print(f"\n🔎 Question : {question}")

    # 1. Filtre Politesse
    st = handle_small_talk(question)
    if st: return st

    # 2. Analyse de l'intention
    is_orientation = is_orientation_question(question)
    search_q = build_search_query(question) if is_orientation else question

    # 3. Récupération des documents
    docs = retrieve(search_q)
    context_text = "\n".join([f"- {d['text']}" for d in docs])

    if not docs:
        return "Je suis navré, je ne trouve pas d'information pertinente dans les documents officiels pour répondre à cette question spécifique."

    # 4. Construction du Prompt (Spécialisé)
    if is_orientation:
        # Prompt "Conseiller RH"
        system_msg = "Tu es un expert RH du CNRS. Aide le candidat à s'orienter en te basant STRICTEMENT sur les textes fournis."
        user_msg = f"""DOCUMENTS :
{context_text}

PROFIL CANDIDAT : {question}

MISSION : Analyse si le profil correspond aux concours mentionnés. Sois nuancé et cite les critères."""
    else:
        # Prompt "Jury Strict" (Classique)
        system_msg = "Tu es un membre de jury de concours CNRS strict. Réponds uniquement à partir du contexte fourni."
        user_msg = f"""CONTEXTE OFFICIEL :
{context_text}

QUESTION : {question}"""

    # 5. Génération (Appel au Modèle Fine-Tuné)
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]

    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=400,
        temperature=0.2, # Un peu de créativité pour l'orientation, strict sinon
        use_cache=True
    )

    response = tokenizer.batch_decode(outputs)[0]

    # Nettoyage de la réponse
    if "assistant\n" in response:
        clean = response.split("assistant\n")[-1].replace("<|im_end|>", "").strip()
    elif "assistant<|end_header_id|>" in response: # Cas Llama 3
        clean = response.split("assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip()
    else:
        clean = response.split(user_msg)[-1].strip()

    return clean

print("🚀 Système RAG Avancé (Notebook Original) rechargé avec succès !")

# Le bon

In [4]:
!pip -q install -U \
  "transformers==4.44.2" \
  "peft==0.12.0" \
  "accelerate==0.33.0" \
  "bitsandbytes==0.43.1" \
  "sentence-transformers==2.6.1" \
  "faiss-cpu==1.9.0" \
  "pandas==2.2.2"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.36.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires n

In [3]:
import os, json
import torch
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import drive


ModuleNotFoundError: No module named 'faiss'

In [4]:
drive.mount("/content/drive")

MODEL_PATH = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Hosni Youssef/Modele_CNRS_FineTuned_QWEN"
JSON_PATH  = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/concours_data.json"

print("MODEL_PATH =", MODEL_PATH)
print("JSON_PATH  =", JSON_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MODEL_PATH = /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Hosni Youssef/Modele_CNRS_FineTuned_QWEN
JSON_PATH  = /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/concours_data.json


# 3) CHARGER TON MODÈLE FINETUNÉ (LOCAL) — version stable

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Device =", device)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 if device == "cuda" else torch.float32,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    quantization_config=bnb,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model.eval()
print("✅ Modèle fine-tuné chargé !")


✅ Device = cuda


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

TypeError: LoraConfig.__init__() got an unexpected keyword argument 'alora_invocation_tokens'

In [ ]:
#
TOP_K = 3
SIMILARITY_THRESHOLD = 0.30
USE_RERANKER = True

# Embeddings
embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

# Reranker
reranker = None
if USE_RERANKER:
    try:
        reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)
        print("✅ Reranker activé.")
    except Exception as e:
        print("⚠️ Reranker non chargé :", e)
        reranker = None

# Documents
with open(JSON_PATH, "r", encoding="utf-8") as f:
    documents = json.load(f)

texts = [d["text"] for d in documents]
embeddings = embedder.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("✅ Index FAISS prêt :", index.ntotal)
